In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Strategies.MultipleMarketsIntensity_class import MultiTradeIntensity as TI
from Math.ti_class import TI_class, VI_class, TR_class
from Math.lm_class import kalman, LinearModel
from Math.accumfeatures import EMA, MA, MSTD, DifferentialEMA, DerivativeEMA
from Strategies.IntensityHawkes_strategy.model_class import HawkesIntensity
from Strategies.IntensityHawkes_strategy.backtest_class import BacktestIB
from Strategies.IntensityHawkes_strategy.strategy_class import StrategyHI, VolumeClass
tol=(1e-1)/2

In [44]:
import sys,os
sys.path.append(r'C:/data/EnergyTrading/Python/')

import cx_Oracle
try:
    cx_Oracle.init_oracle_client(lib_dir=r"C:\Users\andrej\Downloads\instantclient_21_11")
except:
    pass


In [45]:
contracts=[
          'it_dem1','it_deq1','it_dey1'
          ]

In [46]:
start_date='2025-01-01'
end_date='2025-06-30'

In [47]:
data_class = TPData()
data_class.create_connection('OracleSQL')
data_class_pg = TPData()
data_class_pg.create_connection('PostgreSQL')
data_class_tp = TPDataDa()

In [48]:
def get_data_for_contract(contract, start_date, end_date):
    print(contract)
    
    tenor_list =  ['dec'] if contract=='euadec1' else [contract[-2]]
    mkt_list = ['eua'] * len(tenor_list) if contract=='euadec1' else [contract[0:-2]] * len(tenor_list)
    tn_list = [int(contract[-1])]
    prod = 'base'
    venue_list = ['eex']*len(mkt_list)
    # start_date = datetime(2024, 6, 3)
    start_date = start_date
    end_date = end_date

    allwd_broker_ids = [i for i in range(1442)]

    sample_dates = pd.date_range(start_date, end_date, freq='B')

    n = 16
    n_t = 15
    d_t = 1
    date_range_dict = {k.date(): pd.date_range(k, periods=n, freq='B')
                       for k in sample_dates[:-n+1]}

    n_s = 2

    dates = pd.date_range(start_date, end_date, freq='B')
    product_date = [dates.shift(1, freq='B') if t == 'da' else
                    dates.shift(1, freq='D') if t == 'd' else
                    dates.shift(tn, freq='W-MON') if t == 'w' else
                    (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
                    (dates + n_s * dates.freq).shift(tn, freq='YS') if t in ['dec'] else
                    (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
                    for t, tn in zip(tenor_list, tn_list)]

    start_time = time(8, 0, 0)
    end_time = time(18, 0, 0)

    tr_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
    ba_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
    agg_dict = {'price': 'sum', 'volume': 'sum', 'action': 'median',
                'broker_id': 'median', 'count': 'sum'}

    for m, t, n, p_dates in zip(mkt_list, tenor_list, tn_list, product_date):
        df_tr, df_ba = pd.DataFrame([]), pd.DataFrame([])
        series = pd.Series(p_dates, index=dates)
        for p_d, ds in series.groupby(series).groups.items():
            bT = datetime.combine(ds[0], start_time)
            eT = datetime.combine(ds[-1], end_time)
            # Trades
            df_tr_aux = data_class.get_trades(m, t, venue_list, p_d, bT, eT, prod)
            # Filter by broker
            if not allwd_broker_ids or t == 'da':
                pass
            else:
                df_tr_aux = df_tr_aux[df_tr_aux['broker_id'].isin(allwd_broker_ids)]
            # Clean trades
            df_ba_aux = data_class_pg.get_best_ob_data(m, t, venue_list, p_d, bT, eT, prod, None, aonn=False)
            if df_ba_aux.empty:
                df_ba_aux = data_class_tp.get_best_ob_data(m, t, venue_list, p_d, bT, eT, prod, aonn=False)
            df_ba_aux = df_ba_aux.rename(columns={'bidbestprice': 'b_price', 'askbestprice': 'a_price'})
            df_tr_aux = data_class.clean_trades(df_tr_aux, df_ba_aux)
            try:
                df_tr_aux = df_tr_aux.between_time(start_time, end_time)
            except(TypeError):
                pass
            # Group trades
            df_tr_aux['count'] = 1
            #df_tr_aux['price'] *= df_tr_aux['volume']
            #df_tr_aux = df_tr_aux.groupby(df_tr_aux.index).agg(agg_dict)
            #df_tr_aux['price'] /= df_tr_aux['volume']
            df_tr = pd.concat([df_tr, df_tr_aux])
            df_ba = pd.concat([df_ba, df_ba_aux])
            del df_tr_aux, df_ba_aux
        tr_data_dict[m + t + str(n)] = df_tr
        ba_data_dict[m + t + str(n)] = df_ba

    trades2=pd.concat({k: v for k, v in tr_data_dict.items()}, axis=1)
    trades2.columns = ['_'.join([col[-1], col[0]]) for col in trades2.columns]
    trades2.index.name='datetime'
    trades2=trades2[[f'price_{contract}', f'volume_{contract}', f'action_{contract}', f'broker_id_{contract}']]

    ba2=pd.concat({k: v for k, v in ba_data_dict.items()}, axis=1)
    ba2.columns = ['bidbestprice_'+ba2.columns[0][0], 'askbestprice_'+ba2.columns[0][0]]

    trades2.sort_index(inplace=True)
    ba2.sort_index(inplace=True)
    ba2.index.name='datetime'
    trades2.index.name ='datetime'

    products2=[''.join(map(str,(mkt_list+tenor_list+tn_list)))]

    ti_inst = TI(trades2, ba2, products2)

    ti_inst.prepare_data()


    data_raw = ti_inst.data

    data = data_raw[~(data_raw[f'price_{contract}'].notnull() & (data_raw[f'broker_id_{contract}'] != 1441))][[f'price_{contract}', f'volume_{contract}',f'bidbestprice_{contract}',
                      f'askbestprice_{contract}', f'mid_{contract}', f'trade_side_{contract}']].copy()

    # data = data_raw[['price_dew1', 'volume_dew1','bidbestprice_dew1',
    #                    'askbestprice_dew1', 'mid_dew1', 'trade_side_dew1']].copy()

    data.columns = [a.split('_')[0] for a in data.columns]
    data.columns = ['trd_price', 'volume', 'bid_price', 'ask_price', 'mid_price', 'trd_side']

    data.to_parquet(fr's:\Algo\Files\andrej\Data\MassiveMarketMaking\data_{contract}_jan_jun.parquet', engine='fastparquet', compression='gzip')

In [49]:
for contract in contracts:
     get_data_for_contract(contract, start_date, end_date)

it_dem1
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the d